# Formulatrix Mantis

The Mantis is a liquid dispenser from Formulatrix that dispenses through exchangeable chips. PyLabRobot connects to it, identifies it, reads its firmware versions, name and motor limits, locks and unlocks the plate clamp, and reads which chip sits in each input slot. Nothing else in this driver moves the device, switches the pump on or opens a valve.

```{device-card} formulatrix-mantis
```

```{warning}
This driver has run on one Mantis v4, on macOS and on Windows; no v3 has run it yet. `setup()` logs a warning on a firmware set not seen on a Mantis yet, which includes every v3. If you run this guide on a Mantis, please [report the result](../../needs-testing.md).
```

## Properties

| | |
|---|---|
| Generations | v3 and v4, told apart at setup |
| Reads | firmware versions, device name, motor limits |
| Interface | USB, an FTDI bridge (VID 0x0403 / PID 0x6010): interface B for commands, interface A for the RFID reader |
| Plate clamp | lock at one of the unit's levels, waiting for clamping done; unlock |
| RFID | the chip in each input slot: serial, chip type, date of manufacture |
| Unit configuration | the clamp's lock levels and each slot's RFID antenna, from the unit's vendor folder or a saved file; else a recording of its generation |
| Not supported yet | motion and the motor reads, pneumatics and the pressure reads, chips, dispensing |

## How it talks

The Mantis connects over USB through a two-channel FTDI bridge. Interface B carries binary frames to the device's boards: 0 for motion, 10 for pressure, 11 for the absolute position sensors (v4) and 12 for the LC3 board. The driver sends one command at a time and logs every event the device sends while connected. Interface A carries the RFID reader's commands.

At setup the driver reads each board's firmware version. A device whose board 11 answers is a v4; one whose board 11 does not is taken to be a v3.

## Physical setup

1. Connect the Mantis to the computer with its USB cable and switch it on.
2. Install the FTDI dependencies: `pip install "pylabrobot[ftdi]"` (see [Installation](../../getting-started/installation.md)).
3. Close any other software connected to the Mantis.

## Unit configuration

Some of a Mantis's values are set per unit and cannot be read off the device; they sit in the `.config` files of its vendor install, in `Data/Device/Configs`. Give setup that folder once, then save the configuration: the saved file carries the unit's values from then on, and the folder is no longer needed. So far these are the plate clamp's lock levels and each input slot's RFID antenna.

Without them, a recording of a unit of the same generation supplies them, with a warning: the Mantis runs, less accurately. Where the folder and a saved file both hold a value, they must agree, or setup refuses.

In [ ]:
from pylabrobot.formulatrix import Mantis

mantis = Mantis(name="mantis", oem_config_directory="path/to/Data/Device/Configs")

## Setup

`setup()` opens both links, reads the firmware versions and the motor limits, takes the unit's values and sets up the RFID reader. It does not move anything.

In [ ]:
await mantis.setup()
mantis.driver.configuration

### Saved configuration

Save what the device reported and the unit's values. Build the device with that file from then on, `Mantis(name="mantis", declared_configuration_json="my_mantis.json")`, with no folder: setup refuses a device that disagrees with it on the generation or the motor limits.

In [ ]:
mantis.driver.save_configuration(path="my_mantis.json")

### Simulation

`Mantis(simulation=True)` builds a Mantis with nothing behind it; nothing reaches a wire. It takes its configuration, lock levels and RFID antennas from a Mantis v4's recording (`RECORDING_MANTIS_V4`, or a file given as `declared_configuration_json`), its reads answer from it, its plate clamp's state changes at once, and every declared RFID slot reads empty. The device name is not simulated.

In [ ]:
simulated = Mantis(name="simulated_mantis", simulation=True)
await simulated.setup()
simulated.driver.configuration

Stop the simulated Mantis when done with it.

In [ ]:
await simulated.stop()

## Visualizer

Show the Mantis in the 3D visualizer. `start()` serves it and opens it in a browser.

In [ ]:
from pylabrobot.visualizer3D import Viewer3D

viewer = Viewer3D(mantis, name="mantis_hello_world")
await viewer.start()

## Firmware versions

Each board's firmware version, by board, as setup read them; None for a board that did not answer.

In [ ]:
mantis.driver.configuration.firmware_versions

## Name

The name the motion firmware reports, `LiquidHandler` on the v4 we have run. Whether it differs between units is not known; it does not identify this Mantis.

In [ ]:
await mantis.driver.request_name()

## Motors

Motors 0 and 1 drive the arm, motor 2 drives Z. Values are in the device's units. Each motor's limits, as setup read them:

In [ ]:
mantis.driver.configuration.motor_limits

## Plate clamp

The clamp holds the plate on the deck. `lock` releases it, locks it at one of the unit's levels, 1 the weakest, and returns once the device reports clamping done; if it does not, the clamp is released again and an error is raised. `stop()` releases a clamp that does not read unclamping done.

It exists only when the LC3 board (board 12) answered at setup. Its levels are the declared ones:

In [ ]:
mantis.plate_clamp.levels

In [ ]:
await mantis.plate_clamp.lock(level=2)

The clamp's state: 0 idle, 1 stopped, 2 clamping, 3 clamping done, 4 clamping error timeout, 5 unclamping, 6 unclamping done, 7 unclamping error timeout, 8 motor fault. A lock that reads 4 or 8, or an unlock that reads 7 or 8, raises `PlateClampError` at once.

In [ ]:
await mantis.plate_clamp.request_state()

Release the clamp:

In [ ]:
await mantis.plate_clamp.unlock()

## RFID

The RFID reader reads the tag of the chip in each input slot, through that slot's antenna. A slot with no chip reads None. It exists only when the reader answered on interface A at setup; a reader that does not answer leaves setup working, without it. Each slot's antenna, as declared:

In [ ]:
mantis.rfid.slot_to_antenna

Read one slot, or every slot in turn:

In [ ]:
await mantis.rfid.request_tag(slot=1)

In [ ]:
await mantis.rfid.request_tags()

## Stop

Stop the visualizer and close the link.

In [ ]:
await viewer.stop()
await mantis.stop()